# Hamilton PREP Calibration

#### Notebook for Hamilton Prep calibration. Get values from hardware and compare vs calibration commands

## 1. Imports and config

In [ ]:
import sys
import logging
from asyncio import sleep

from pylabrobot.liquid_handling.backends.hamilton.prep_backend import PrepBackend
from pylabrobot.liquid_handling.backends.hamilton.prep_commands import (
    ChannelIndex,
    format_calibration_diff,
)
from pylabrobot.liquid_handling.backends.hamilton.tcp_backend import HamiltonTCPClient
from pylabrobot.liquid_handling import LiquidHandler
from pylabrobot.resources import Coordinate, hamilton_96_tiprack_50uL_NTR
from pylabrobot.resources.hamilton import PrepDeck
from pylabrobot.visualizer import Visualizer

logging.getLogger("pylabrobot").setLevel(logging.INFO)
logging.getLogger("pylabrobot").handlers.clear()
handler = logging.StreamHandler(sys.stdout)
handler.setFormatter(logging.Formatter("%(levelname)s - %(message)s"))
logging.getLogger("pylabrobot").addHandler(handler)

async def get_methods_from_interface(interface):
  pool, reg = await backend.client.introspect(interface)
  print(f"Command signatures on {interface} ({len(reg.methods)} methods):\n")
  for m in reg.methods:
      print(f"  {m.get_signature_string(reg)}")
  return pool, reg

HOST = "192.168.100.102"
PORT = 2000

## 2. Deck layout and visualizer

In [ ]:
deck = PrepDeck(name="deck")

visualizer = Visualizer(deck, open_browser=False)
await visualizer.setup()

## 3. Backend and liquid handler setup

In [ ]:
backend = PrepBackend(host=HOST, port=PORT)
lh = LiquidHandler(backend=backend, deck=deck)
await lh.setup(smart=True, force_initialize=False)

# Deck, waste, and other config data from setup can be accessed here.
config = backend._config
print("Instrument configuration:")
print(f"  Deck bounds: {config.deck_bounds}")
print(f"  Has enclosure: {config.has_enclosure}")
print(f"  Safe speeds enabled: {config.safe_speeds_enabled}")
print(f"  Default traverse height: {config.default_traverse_height}")
print(f"  Number of channels: {config.num_channels}")
print(f"  Has MPH: {config.has_mph}")

print("\nCalibration sites:")
for cal in await backend.get_calibration_site_definitions():
    print(cal)

In [ ]:
# Uncomment to print firmware tree
# await backend.print_firmware_tree()

# Uncomment to get methods from calibration interface
# pool, reg = await get_methods_from_interface("MLPrepRoot.MLPrepCalibration")


## 4. Calibration commands (MLPrepCalibration)

Test the calibration query and action commands exposed via `PrepBackend`.

In [ ]:
# Read current calibration values without entering calibration mode/session.
cal_values = await backend.read_calibration_values()
print(cal_values)

### Enter Calibration mode and run calibration commands

In [ ]:
# Enter managed calibration session before running calibration commands.
cal_session = backend.calibration_session(report_after_command=True, report_scope="related")
await cal_session.start()
print("Calibration session started.")

In [ ]:
# Calibrate X/Y/Z axes for channel 0 at calibration site 0
z_result = await cal_session.calibrate_z_axis(site_index=0, channel=ChannelIndex.RearChannel)
print(f"Z-axis offset (RearChannel, site 0): {z_result.result}")
print(format_calibration_diff(z_result.diff))

y_result = await cal_session.calibrate_y_axis(site_index=0, channel=ChannelIndex.RearChannel)
print(f"Y-axis offset (RearChannel, site 0): {y_result.result}")
print(format_calibration_diff(y_result.diff))

x_result = await cal_session.calibrate_x_axis(site_index=0, channel=ChannelIndex.RearChannel)
print(f"X-axis offset (RearChannel, site 0): {x_result.result}")
print(format_calibration_diff(x_result.diff))


### Squeeze drive calibration: Add tips before this step

In [ ]:
# CalibrateSqueezeTips using NTR tip rack — A1 (rear), B1 (front)
tip_rack = deck[5] = hamilton_96_tiprack_50uL_NTR(name="ntr_50", with_tips=True)
sq_result = await cal_session.calibrate_squeeze_tips(tip_spots=tip_rack["A1:B1"])
print(f"Squeeze tip positions: {sq_result.result}")
print(format_calibration_diff(sq_result.diff))

In [ ]:
# Exit calibration mode, does not save values (TODO: Add set_calibration_values to write to hardware)
await cal_session.end(save=False)
print("Calibration session cancelled.")

In [ ]:
await backend.spread()

## 5. Teardown

In [ ]:
await backend.park()
await lh.stop()